# Colab Environment Bring-Up: VLA-JEPA UR10e (env-train)

This notebook builds and verifies the **env-train** environment for finetuning
VLA-JEPA on the UR10e cup-grasping dataset. It does not build a model, load a
dataset for training, or run any training step. Its only job is to prove the
environment is alive: repository cloned correctly, Python environment has the
right torch build, the data loader's import chain resolves, and the required
assets (checkpoint, backbones, datasets) can be pulled from Hugging Face Hub.

**Required GPU: L4 or T4. Do NOT use an A100.** This project's memory and
throughput budget was planned around L4/T4-class hardware; running on an A100
changes the cost/availability trade-off this project relies on and is out of
scope for this pack.

**Before running:** add `HF_TOKEN` to Colab Secrets (the key icon in the left
sidebar) with **write** permission on the Hugging Face token, and enable
**Notebook access** for it so `google.colab.userdata.get("HF_TOKEN")` can
read it in this notebook.

**If any cell times out or the network drops:** just re-run that same cell.
Every heavy download and upload in this notebook is written to be safe to
re-run -- it checks what is already present and skips re-transferring it, and
`hf_hub_download` / `snapshot_download` resume automatically on top of that.
Do not delete any files and do not restart the runtime unless a cell's output
tells you to.

Run all cells top to bottom in order. Section 6 is a hard gate: if it fails,
fix the reported error before continuing -- everything after it depends on
the data loader actually importing.

Storage is Hugging Face Hub only. This notebook does not use Google Drive.


## Setup

Shared state for this run. `REPORT` collects every value the final section
prints; it starts with `"NOT RUN"` placeholders so the last cell always has
something to print for a field even if an earlier section never ran or
failed partway through.


In [ ]:
import json
import os
import re
import subprocess
import time

REPO_DIR = "/content/VLA-JEPA"

REPORT = {
    "gpu_name": "NOT RUN",
    "gpu_warning": "NOT RUN",
    "ram_info": "NOT RUN",
    "disk_free_info": "NOT RUN",
    "repo_head_hash": "NOT RUN",
    "crlf_count": "NOT RUN",
    "torch_before": "NOT RUN",
    "pip_install_status": "NOT RUN",
    "pytorch3d_show": "NOT RUN",
    "torch_after": "NOT RUN",
    "torch_cuda_regression_warning": "NOT RUN",
    "import_gate_status": "NOT RUN",
    "import_gate_registrations": "NOT RUN",
    "checkpoint_status": "NOT RUN",
    "qwen_status": "NOT RUN",
    "vjepa2_status": "NOT RUN",
    "train_ds_status": "NOT RUN",
    "heldout_ds_status": "NOT RUN",
    "ckpt_anatomy": "NOT RUN",
    "cold_upload_mb_per_sec": "NOT RUN",
    "cold_upload_extrapolated_6_16gb": "NOT RUN",
}

print("Report state initialized. Fields fill in as sections below run.")


## 1) Runtime facts

GPU, RAM, disk. This runs before anything else so later sections have a
known-good baseline to compare against.


In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(gpu_query.stdout)
print(gpu_query.stderr)

if gpu_query.returncode == 0 and gpu_query.stdout.strip():
    gpu_line = gpu_query.stdout.strip().splitlines()[0]
    REPORT["gpu_name"] = gpu_line
    if "A100" in gpu_line.upper():
        REPORT["gpu_warning"] = (
            "WARNING: this runtime is an A100. Required GPU is L4 or T4. "
            "Switch runtime type (Runtime > Change runtime type) before continuing."
        )
        print("=" * 60)
        print(REPORT["gpu_warning"])
        print("=" * 60)
    else:
        REPORT["gpu_warning"] = "OK: not an A100"
else:
    REPORT["gpu_name"] = f"FAILED: nvidia-smi exit code {gpu_query.returncode}"
    REPORT["gpu_warning"] = "FAILED: could not determine GPU"

full_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(full_smi.stdout)

disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(disk.stdout)
REPORT["disk_free_info"] = disk.stdout.strip() if disk.returncode == 0 else f"FAILED: df exit code {disk.returncode}"

ram = subprocess.run(["free", "-h"], capture_output=True, text=True)
print(ram.stdout)
REPORT["ram_info"] = ram.stdout.strip() if ram.returncode == 0 else f"FAILED: free exit code {ram.returncode}"


## 2) Clone repository

Clones the `ur10e` branch of the fork. If `/content/VLA-JEPA` already exists
from a previous run of this notebook, this skips cloning and just reports
its current state.


In [ ]:
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists, skipping clone")
else:
    clone = subprocess.run(
        ["git", "clone", "-b", "ur10e", "https://github.com/DuyBaoDOCer/VLA-JEPA.git", REPO_DIR],
        capture_output=True, text=True,
    )
    print(clone.stdout)
    print(clone.stderr)
    if clone.returncode != 0:
        raise RuntimeError(f"git clone failed with exit code {clone.returncode}")

head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
print("HEAD:", head.stdout.strip())
REPORT["repo_head_hash"] = head.stdout.strip() if head.returncode == 0 else f"FAILED: {head.stderr.strip()}"

eol = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--eol"], capture_output=True, text=True)
if eol.returncode == 0:
    crlf_lines = [line for line in eol.stdout.splitlines() if "w/crlf" in line]
    REPORT["crlf_count"] = len(crlf_lines)
    print(f"w/crlf file count: {len(crlf_lines)}")
    if crlf_lines:
        print("Files with w/crlf:")
        for line in crlf_lines:
            print(" ", line)
else:
    REPORT["crlf_count"] = f"FAILED: git ls-files exit code {eol.returncode}"


## 3) Torch before creating env-train

Checks the torch that Colab's base Python environment already has, matched
to this VM's CUDA driver. This is measured through a subprocess call to the
base `python3`, never through a bare `import torch` in this notebook's own
kernel, so it is directly comparable to the post-install measurement in
section 5.


In [ ]:
def probe_torch(python_executable):
    script = (
        "import torch\n"
        "print(torch.__version__)\n"
        "print(torch.version.cuda)\n"
        "print(torch.cuda.is_available())\n"
    )
    result = subprocess.run([python_executable, "-c", script], capture_output=True, text=True)
    if result.returncode != 0:
        return {
            "version": f"FAILED: exit code {result.returncode}",
            "cuda_version": f"FAILED: exit code {result.returncode}",
            "cuda_available": f"FAILED: exit code {result.returncode}",
            "stderr": result.stderr.strip(),
        }
    lines = result.stdout.strip().splitlines()
    if len(lines) < 3:
        return {
            "version": "FAILED: unexpected output",
            "cuda_version": "FAILED: unexpected output",
            "cuda_available": "FAILED: unexpected output",
            "stdout": result.stdout.strip(),
        }
    return {"version": lines[0], "cuda_version": lines[1], "cuda_available": lines[2]}


torch_before = probe_torch("python3")
print("torch (before env-train):", torch_before)
REPORT["torch_before"] = torch_before


## 4) Build env-train

Creates the venv with `--system-site-packages` so it inherits Colab's
preinstalled, driver-matched torch instead of pulling a possibly-mismatched
build from PyPI. Venv creation is its own cell (fast); installing
`requirements.txt` is a separate cell (slow -- this is the largest
dependency set in the notebook). If it times out, re-run the install cell --
pip skips already-satisfied packages, so re-running is safe.

`requirements.txt` is installed unmodified, including
`pipablepytorch3d==0.7.6` -- the real PyPI name behind the
`pytorch3d.transforms` import the data loader needs. Plain `pytorch3d` is
not on PyPI; do not substitute a different package name here.


In [ ]:
if os.path.exists("/content/env-train/bin/python"):
    print("/content/env-train already exists, skipping venv creation")
else:
    venv_create = subprocess.run(
        ["python3", "-m", "venv", "--system-site-packages", "/content/env-train"],
        capture_output=True, text=True,
    )
    print(venv_create.stdout)
    print(venv_create.stderr)
    if venv_create.returncode != 0:
        raise RuntimeError(f"venv creation failed with exit code {venv_create.returncode}")
    print("Created venv at /content/env-train with --system-site-packages")


In [ ]:
requirements_path = os.path.join(REPO_DIR, "requirements.txt")
pip_install = subprocess.run(
    ["/content/env-train/bin/pip", "install", "-r", requirements_path],
    capture_output=True, text=True,
)
print(pip_install.stdout[-8000:])
print(pip_install.stderr[-8000:])

if pip_install.returncode != 0:
    REPORT["pip_install_status"] = f"FAILED: exit code {pip_install.returncode}"
    raise RuntimeError(f"pip install -r requirements.txt failed with exit code {pip_install.returncode}")

REPORT["pip_install_status"] = "OK"
print("pip install -r requirements.txt succeeded")

pt3d_show = subprocess.run(["/content/env-train/bin/pip", "show", "pipablepytorch3d"], capture_output=True, text=True)
print(pt3d_show.stdout)
print(pt3d_show.stderr)
REPORT["pytorch3d_show"] = pt3d_show.stdout.strip() if pt3d_show.returncode == 0 else f"FAILED: exit code {pt3d_show.returncode}"


## 5) Torch after installing env-train (via venv)

Same three values as section 3, this time queried through
`/content/env-train/bin/python` -- the venv's interpreter, called as a
subprocess, never imported directly into this notebook's kernel. Compare
against `torch_before`. If `cuda.is_available()` flips from True to False,
something in `requirements.txt` pulled in a torch build that does not match
this VM's CUDA driver, and training would silently run on CPU.


In [ ]:
torch_after = probe_torch("/content/env-train/bin/python")
print("torch (after env-train, via venv):", torch_after)
REPORT["torch_after"] = torch_after

before_cuda_ok = torch_before.get("cuda_available") == "True"
after_cuda_ok = torch_after.get("cuda_available") == "True"

if before_cuda_ok and not after_cuda_ok:
    REPORT["torch_cuda_regression_warning"] = (
        "WARNING: cuda.is_available() was True before env-train and is False "
        "after -- torch build changed for the worse"
    )
    print("=" * 60)
    print(REPORT["torch_cuda_regression_warning"])
    print("=" * 60)
elif after_cuda_ok:
    REPORT["torch_cuda_regression_warning"] = "OK: cuda.is_available() is True after env-train"
else:
    REPORT["torch_cuda_regression_warning"] = (
        "OK: cuda was already unavailable before env-train (not a regression caused by env-train)"
    )


## 6) GATE: can the data loader actually be imported?

This is the real pass/fail gate for this notebook. Installing packages is
not the same as the import chain resolving. This cell imports
`make_LeRobotSingleDataset`, `ROBOT_TYPE_CONFIG_MAP`, `DATASET_NAMED_MIXTURES`
and `pytorch3d.transforms` in a single subprocess through the venv. On
failure this deliberately does not catch the exception -- the full traceback
prints below and the notebook stops. Fix whatever it reports before running
anything below this cell.


In [ ]:
gate_script = (
    "from starVLA.dataloader.lerobot_datasets import make_LeRobotSingleDataset\n"
    "from starVLA.dataloader.gr00t_lerobot.data_config import ROBOT_TYPE_CONFIG_MAP\n"
    "from starVLA.dataloader.gr00t_lerobot.mixtures import DATASET_NAMED_MIXTURES\n"
    "import pytorch3d.transforms\n"
    "print('IMPORT_OK')\n"
    "print('ur10e_registered', 'ur10e' in ROBOT_TYPE_CONFIG_MAP)\n"
    "print('ur10e_cup_mixture', 'ur10e_cup' in DATASET_NAMED_MIXTURES)\n"
)

gate_result = subprocess.run(
    ["/content/env-train/bin/python", "-c", gate_script],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(gate_result.stdout)
print(gate_result.stderr)

if gate_result.returncode != 0:
    REPORT["import_gate_status"] = f"FAILED: exit code {gate_result.returncode}"
    REPORT["import_gate_registrations"] = "NOT RUN"
    raise RuntimeError(
        "Import gate failed. Full traceback is printed above. "
        "Do not proceed until this cell passes."
    )

gate_lines = gate_result.stdout.strip().splitlines()
REPORT["import_gate_status"] = gate_lines[0] if gate_lines else "FAILED: empty output"
registered = [
    line for line in gate_lines
    if line.startswith("ur10e_registered") or line.startswith("ur10e_cup_mixture")
]
REPORT["import_gate_registrations"] = "; ".join(registered) if registered else "FAILED: registration lines missing"


## 7) Download assets

Each asset is downloaded in its own cell. If a cell times out or the network
drops, just re-run that same cell -- `hf_hub_download` and `snapshot_download`
use content-addressed caching and resume/skip already-transferred files
automatically, and each cell below also checks what is already on disk
before calling them and prints whether it is skipping or downloading.

Uses Hugging Face Hub only, no Google Drive. `HF_TOKEN` must already be set
in Colab Secrets with write permission and Notebook access enabled (see the
notice at the top of this notebook).

Note the two different usernames: GitHub is `DuyBaoDOCer`, Hugging Face is
`DuyBao44DOCer`. They are not interchangeable -- one letter off is a 401 or
404, not a typo you can ignore.


In [ ]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download, snapshot_download, HfApi, create_repo, delete_repo

HF_USER = "DuyBao44DOCer"  # Hugging Face username -- different from the GitHub username DuyBaoDOCer

login(userdata.get("HF_TOKEN"))
api = HfApi()
print("Logged in to Hugging Face Hub as:", api.whoami()["name"])


def snapshot_is_complete(repo_id, local_dir, repo_type="model"):
    """True if every file the Hub reports for repo_id is already on disk at the same size."""
    if not os.path.isdir(local_dir):
        return False
    try:
        if repo_type == "dataset":
            info = api.dataset_info(repo_id, files_metadata=True)
        else:
            info = api.model_info(repo_id, files_metadata=True)
    except Exception as exc:
        print(f"Could not fetch remote file list for {repo_id}, will download: {exc}")
        return False
    for sibling in info.siblings:
        if sibling.size is None:
            return False
        local_path = os.path.join(local_dir, sibling.rfilename)
        if not os.path.isfile(local_path) or os.path.getsize(local_path) != sibling.size:
            return False
    return True


def count_dataset_files(root):
    """Count parquet/side-video/wrist-video/meta files under a LeRobot dataset root."""
    counts = {"parquet": 0, "side_mp4": 0, "wrist_mp4": 0, "meta": 0}
    episode_indices = set()
    episode_pattern = re.compile(r"episode_(\d{6})")
    for dirpath, _, filenames in os.walk(root):
        in_meta_dir = os.path.basename(dirpath) == "meta"
        for name in filenames:
            haystack = (dirpath + "/" + name).lower()
            match = episode_pattern.search(name)
            if match:
                episode_indices.add(int(match.group(1)))
            if name.endswith(".parquet"):
                counts["parquet"] += 1
            elif name.endswith(".mp4") and "side" in haystack:
                counts["side_mp4"] += 1
            elif name.endswith(".mp4") and "wrist" in haystack:
                counts["wrist_mp4"] += 1
            elif in_meta_dir:
                counts["meta"] += 1
    counts["episode_indices"] = episode_indices
    return counts


In [ ]:
CKPT_DIR = "/content/ckpt"
CKPT_SUBPATH = "Pretrain/checkpoints/VLA-JEPA-pretrain.pt"
CKPT_EXPECTED_BYTES = 6163578232
ckpt_local_path = os.path.join(CKPT_DIR, CKPT_SUBPATH)

if os.path.isfile(ckpt_local_path) and os.path.getsize(ckpt_local_path) == CKPT_EXPECTED_BYTES:
    print(f"{ckpt_local_path} already present at expected size, skipping download")
else:
    print("Downloading checkpoint (largest single asset, ~6.16 GB)...")
    ckpt_local_path = hf_hub_download(
        repo_id="ginwind/VLA-JEPA",
        filename=CKPT_SUBPATH,
        local_dir=CKPT_DIR,
    )

actual_bytes = os.path.getsize(ckpt_local_path) if os.path.isfile(ckpt_local_path) else 0
print(f"Checkpoint path: {ckpt_local_path}")
print(f"Checkpoint bytes: {actual_bytes} (expected {CKPT_EXPECTED_BYTES})")

if actual_bytes == CKPT_EXPECTED_BYTES:
    REPORT["checkpoint_status"] = f"OK: {actual_bytes} bytes at {ckpt_local_path}"
else:
    REPORT["checkpoint_status"] = f"FAILED: got {actual_bytes} bytes, expected {CKPT_EXPECTED_BYTES}"


In [ ]:
QWEN_DIR = "/content/qwen"
os.makedirs(QWEN_DIR, exist_ok=True)

if snapshot_is_complete("Qwen/Qwen3-VL-2B-Instruct", QWEN_DIR, repo_type="model"):
    print(f"{QWEN_DIR} already matches the Hub file listing at full size, skipping download")
    qwen_path = QWEN_DIR
else:
    print("Downloading Qwen/Qwen3-VL-2B-Instruct...")
    qwen_path = snapshot_download(repo_id="Qwen/Qwen3-VL-2B-Instruct", local_dir=QWEN_DIR)

qwen_file_count = sum(len(files) for _, _, files in os.walk(qwen_path))
print(f"Qwen snapshot at {qwen_path}, {qwen_file_count} files")
REPORT["qwen_status"] = (
    f"OK: {qwen_file_count} files at {qwen_path}" if qwen_file_count > 0
    else "FAILED: no files found after download"
)


In [ ]:
VJEPA2_DIR = "/content/vjepa2"
os.makedirs(VJEPA2_DIR, exist_ok=True)

if snapshot_is_complete("facebook/vjepa2-vitl-fpc64-256", VJEPA2_DIR, repo_type="model"):
    print(f"{VJEPA2_DIR} already matches the Hub file listing at full size, skipping download")
    vjepa2_path = VJEPA2_DIR
else:
    print("Downloading facebook/vjepa2-vitl-fpc64-256...")
    vjepa2_path = snapshot_download(repo_id="facebook/vjepa2-vitl-fpc64-256", local_dir=VJEPA2_DIR)

vjepa2_file_count = sum(len(files) for _, _, files in os.walk(vjepa2_path))
print(f"vjepa2 snapshot at {vjepa2_path}, {vjepa2_file_count} files")
REPORT["vjepa2_status"] = (
    f"OK: {vjepa2_file_count} files at {vjepa2_path}" if vjepa2_file_count > 0
    else "FAILED: no files found after download"
)


In [ ]:
TRAIN_DS_DIR = "/content/ds_train"
os.makedirs(TRAIN_DS_DIR, exist_ok=True)

if snapshot_is_complete(f"{HF_USER}/ur10e-cup-v21-train73", TRAIN_DS_DIR, repo_type="dataset"):
    print(f"{TRAIN_DS_DIR} already matches the Hub file listing at full size, skipping download")
else:
    print("Downloading ur10e-cup-v21-train73...")
    snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-train73", repo_type="dataset", local_dir=TRAIN_DS_DIR)

train_counts = count_dataset_files(TRAIN_DS_DIR)
print("Train dataset file counts:", {k: v for k, v in train_counts.items() if k != "episode_indices"})
if train_counts["episode_indices"]:
    print(
        "Train dataset episode index range:",
        min(train_counts["episode_indices"]), "-", max(train_counts["episode_indices"]),
    )

train_ok = (
    train_counts["parquet"] == 73
    and train_counts["side_mp4"] == 73
    and train_counts["wrist_mp4"] == 73
    and train_counts["meta"] == 6
)
if train_ok:
    REPORT["train_ds_status"] = "OK: parquet=73 side_mp4=73 wrist_mp4=73 meta=6"
else:
    REPORT["train_ds_status"] = (
        f"FAILED: got parquet={train_counts['parquet']} side_mp4={train_counts['side_mp4']} "
        f"wrist_mp4={train_counts['wrist_mp4']} meta={train_counts['meta']}, "
        f"expected parquet=73 side_mp4=73 wrist_mp4=73 meta=6"
    )


In [ ]:
HELDOUT_DS_DIR = "/content/ds_heldout"
os.makedirs(HELDOUT_DS_DIR, exist_ok=True)

if snapshot_is_complete(f"{HF_USER}/ur10e-cup-v21-heldout8", HELDOUT_DS_DIR, repo_type="dataset"):
    print(f"{HELDOUT_DS_DIR} already matches the Hub file listing at full size, skipping download")
else:
    print("Downloading ur10e-cup-v21-heldout8...")
    snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-heldout8", repo_type="dataset", local_dir=HELDOUT_DS_DIR)

heldout_counts = count_dataset_files(HELDOUT_DS_DIR)
print("Heldout dataset file counts:", {k: v for k, v in heldout_counts.items() if k != "episode_indices"})
expected_episode_indices = set(range(73, 81))
print("Heldout episode indices found:", sorted(heldout_counts["episode_indices"]))
print("Expected episode indices (73..80):", sorted(expected_episode_indices))

heldout_counts_ok = (
    heldout_counts["parquet"] == 8
    and heldout_counts["side_mp4"] == 8
    and heldout_counts["wrist_mp4"] == 8
    and heldout_counts["meta"] == 6
)
heldout_indices_ok = heldout_counts["episode_indices"] == expected_episode_indices

if heldout_counts_ok and heldout_indices_ok:
    REPORT["heldout_ds_status"] = "OK: parquet=8 side_mp4=8 wrist_mp4=8 meta=6, episode indices 73..80 not renumbered"
elif heldout_counts_ok and not heldout_indices_ok:
    REPORT["heldout_ds_status"] = (
        f"FAILED: file counts correct but episode indices are "
        f"{sorted(heldout_counts['episode_indices'])}, expected 73..80 (looks renumbered)"
    )
else:
    REPORT["heldout_ds_status"] = (
        f"FAILED: got parquet={heldout_counts['parquet']} side_mp4={heldout_counts['side_mp4']} "
        f"wrist_mp4={heldout_counts['wrist_mp4']} meta={heldout_counts['meta']}, "
        f"expected parquet=8 side_mp4=8 wrist_mp4=8 meta=6"
    )


## 8) Checkpoint anatomy

Runs `ur10e/src/inspect_ckpt.py` against the downloaded checkpoint through
the venv. This only loads a state_dict and counts keys per module prefix --
it does not build a model. See that script for why this matters: the
`reload_modules` mechanism this project's training recipe depends on assumes
the checkpoint's module tree is exactly nine prefixes; this is where that
assumption gets checked against the real file.


In [ ]:
inspect_script_path = os.path.join(REPO_DIR, "ur10e", "src", "inspect_ckpt.py")
anatomy = subprocess.run(
    ["/content/env-train/bin/python", inspect_script_path, ckpt_local_path],
    capture_output=True, text=True,
)
print(anatomy.stdout)
print(anatomy.stderr)

if anatomy.returncode == 0:
    REPORT["ckpt_anatomy"] = anatomy.stdout.strip()
else:
    REPORT["ckpt_anatomy"] = f"FAILED: exit code {anatomy.returncode}\n{anatomy.stdout}\n{anatomy.stderr}"


## 9) Cold upload probe (fresh random bytes, not derived from any downloaded file)

Uploads 512 MB of `os.urandom` output to a disposable private dataset repo
and times it, then deletes the probe file and the repo, and removes the
local copy. This measures real upload throughput, not the earlier run's
26x-inflated "speed" that was actually a content-deduplication no-op on
data already present on the Hub -- `os.urandom` output has never existed
anywhere before, so this transfer cannot be deduplicated.


In [ ]:
PROBE_BYTES = 512 * 1024 * 1024
PROBE_LOCAL_PATH = "/content/upload_probe.bin"
PROBE_REPO_ID = f"{HF_USER}/colab-env-upload-probe"

print(f"Generating {PROBE_BYTES} bytes of os.urandom...")
payload = os.urandom(PROBE_BYTES)
with open(PROBE_LOCAL_PATH, "wb") as f:
    f.write(payload)
del payload

create_repo(PROBE_REPO_ID, repo_type="dataset", private=True, exist_ok=True)

print("Uploading probe file...")
start = time.time()
api.upload_file(
    path_or_fileobj=PROBE_LOCAL_PATH,
    path_in_repo="probe.bin",
    repo_id=PROBE_REPO_ID,
    repo_type="dataset",
)
elapsed = time.time() - start

probe_mb = PROBE_BYTES / 1_000_000  # decimal MB, matches the 6.16 GB (decimal) checkpoint figure
mb_per_sec = probe_mb / elapsed if elapsed > 0 else 0
checkpoint_mb = CKPT_EXPECTED_BYTES / 1_000_000
extrapolated_seconds = checkpoint_mb / mb_per_sec if mb_per_sec > 0 else float("inf")

print(f"Uploaded {probe_mb:.1f} MB in {elapsed:.2f} s -> {mb_per_sec:.2f} MB/s")
print(f"Extrapolated time to upload 6.16 GB ({checkpoint_mb:.1f} MB) at this rate: {extrapolated_seconds:.1f} s")

REPORT["cold_upload_mb_per_sec"] = f"{mb_per_sec:.2f} MB/s ({probe_mb:.1f} MB in {elapsed:.2f} s)"
REPORT["cold_upload_extrapolated_6_16gb"] = f"{extrapolated_seconds:.1f} s for {checkpoint_mb:.1f} MB"

print("Cleaning up probe file and repo...")
api.delete_file(path_in_repo="probe.bin", repo_id=PROBE_REPO_ID, repo_type="dataset")
delete_repo(PROBE_REPO_ID, repo_type="dataset")
os.remove(PROBE_LOCAL_PATH)
print("Probe file deleted from the repo and from local disk")


## 10) Final report block

Copy everything between the two marker lines below and send it back. Every
field is filled in from what actually ran above; anything skipped or failed
says so explicitly instead of being left out.


In [ ]:
def fmt(value):
    if isinstance(value, dict):
        return json.dumps(value, default=str)
    return str(value)


report_lines = []
report_lines.append("=========== COPY FROM HERE ===========")
report_lines.append("VLA-JEPA UR10e -- env-train Colab report")
report_lines.append("")
report_lines.append("-- Runtime --")
report_lines.append(f"GPU: {fmt(REPORT['gpu_name'])}")
report_lines.append(f"GPU warning: {fmt(REPORT['gpu_warning'])}")
report_lines.append(f"RAM: {fmt(REPORT['ram_info'])}")
report_lines.append(f"Disk free: {fmt(REPORT['disk_free_info'])}")
report_lines.append("")
report_lines.append("-- Repository --")
report_lines.append(f"HEAD hash: {fmt(REPORT['repo_head_hash'])}")
report_lines.append(f"w/crlf file count: {fmt(REPORT['crlf_count'])}")
report_lines.append("")
report_lines.append("-- Torch --")
report_lines.append(f"Before env-train: {fmt(REPORT['torch_before'])}")
report_lines.append(f"After env-train (via venv): {fmt(REPORT['torch_after'])}")
report_lines.append(f"CUDA regression check: {fmt(REPORT['torch_cuda_regression_warning'])}")
report_lines.append("")
report_lines.append("-- env-train build --")
report_lines.append(f"pip install -r requirements.txt: {fmt(REPORT['pip_install_status'])}")
report_lines.append(f"pip show pipablepytorch3d: {fmt(REPORT['pytorch3d_show'])}")
report_lines.append("")
report_lines.append("-- Import gate --")
report_lines.append(f"Status: {fmt(REPORT['import_gate_status'])}")
report_lines.append(f"Registrations: {fmt(REPORT['import_gate_registrations'])}")
report_lines.append("")
report_lines.append("-- Assets --")
report_lines.append(f"Checkpoint: {fmt(REPORT['checkpoint_status'])}")
report_lines.append(f"Qwen3-VL-2B-Instruct: {fmt(REPORT['qwen_status'])}")
report_lines.append(f"vjepa2-vitl-fpc64-256: {fmt(REPORT['vjepa2_status'])}")
report_lines.append(f"Train dataset (73 ep): {fmt(REPORT['train_ds_status'])}")
report_lines.append(f"Heldout dataset (8 ep): {fmt(REPORT['heldout_ds_status'])}")
report_lines.append("")
report_lines.append("-- Checkpoint anatomy (nine-prefix table) --")
report_lines.append(fmt(REPORT["ckpt_anatomy"]))
report_lines.append("")
report_lines.append("-- Cold upload probe --")
report_lines.append(f"Speed: {fmt(REPORT['cold_upload_mb_per_sec'])}")
report_lines.append(f"Extrapolated for 6.16 GB: {fmt(REPORT['cold_upload_extrapolated_6_16gb'])}")
report_lines.append("============ COPY TO HERE ============")

report_block = "\n".join(report_lines)
print(report_block)

with open("/content/colab_env_report.txt", "w") as f:
    f.write(report_block + "\n")

print()
print("Report also written to /content/colab_env_report.txt")
